In [15]:
import pandas as pd
import json
from typing import List, Dict

In [16]:
squad_df = pd.read_csv("raw_data/archive/2022_FIFA_World_Cup_squads.csv")

In [17]:
squad_df.columns

Index(['No.', 'Pos.', 'Player', 'Date of birth (age)', 'Caps', 'Goals', 'Club',
       'Country'],
      dtype='str')

In [18]:
mapped_teams = {
    "prem_matches team": "wc team",
    "premier_league": {
        "Manchester Utd": "Manchester United",
        "Newcastle Utd": "Newcastle United",
        "Tottenham": "Tottenham Hotspur",
        "Brighton": "Brighton & Hove Albion",
        "Wolves": "Wolverhampton Wanderers",
        "West Ham": "West Ham United"
    },
    "laliga": {
        "Cadiz": "Cádiz",
        "Celta": "Celta Vigo",
        "Betis": "Real Betis",
        "Espanol": "Espanyol",
        "Ath Bilbao": "Athletic Bilbao",
        "Sociedad": "Real Sociedad",
        "Ath Madrid": "Atlético Madrid",
        "Vallecano": "Rayo Vallecano"
    },
    "bundesliga": {
        "Dortmund": "Borussia Dortmund",
        'Ein Frankfurt': 'Eintracht Frankfurt',
        'FC Koln': '1. FC Köln',
        "Hannover": "Hannover 96",
        "Hertha": "Hertha BSC",
        'Leverkusen': 'Bayer Leverkusen',
        "M'gladbach": 'Borussia Mönchengladbach',
        "Mainz": 'Mainz 05',
        "Wolfsburg": 'VfL Wolfsburg',
        "Stuttgart": 'VfB Stuttgart',
        "Hoffenheim": "1899 Hoffenheim",
        "Freiburg": "SC Freiburg",
        "Augsburg": "FC Augsburg"
    },
    "serie_a": {
        "Inter": "Internazionale",
        "Verona": "Hellas Verona"
    },
    "ligue_1": {
        "Paris SG": "Paris Saint-Germain",
    },
}

In [19]:
squad_df[squad_df["Country"] == "Ecuador"]["Player"]

0               Hernán Galíndez
1                  Félix Torres
2                Piero Hincapié
3               Robert Arboleda
4                José Cifuentes
5                 William Pacho
6              Pervis Estupiñán
7                 Carlos Gruezo
8               Ayrton Preciado
9                Romario Ibarra
10              Michael Estrada
11               Moisés Ramírez
12    Enner Valencia*(captain)*
13               Xavier Arreaga
14                   Ángel Mena
15             Jeremy Sarmiento
16              Ángelo Preciado
17               Diego Palacios
18                Gonzalo Plata
19                 Sebas Méndez
20                  Alan Franco
21          Alexander Domínguez
22               Moisés Caicedo
23             Djorkaeff Reasco
24               Jackson Porozo
25              Kevin Rodríguez
Name: Player, dtype: str

In [20]:
prem_df = pd.read_csv("raw_data/prem_matches.csv")
prem_df.columns

Index(['Unnamed: 0', 'Season', 'Date', 'Home', 'xG', 'Home Goals',
       'Away Goals', 'xG.1', 'Away', 'Attendance', 'Venue'],
      dtype='str')

In [21]:
prem_df[prem_df["Season"] == "2021/2022"]["Home"].unique().tolist()

['Brentford',
 'Manchester Utd',
 'Leicester City',
 'Burnley',
 'Chelsea',
 'Watford',
 'Everton',
 'Norwich City',
 'Newcastle Utd',
 'Tottenham',
 nan,
 'Liverpool',
 'Aston Villa',
 'Manchester City',
 'Leeds United',
 'Crystal Palace',
 'Brighton',
 'Wolves',
 'Southampton',
 'Arsenal',
 'West Ham']

In [22]:
def check_teams_in_wc(teams, league="premier_league"):
    wc_teams = squad_df["Club"].unique().tolist()
    for team in teams:
        team = mapped_teams[league].get(team, team)
        if team not in wc_teams:
            print(team)


In [23]:
laliga_df = pd.read_csv("raw_data/ligue_1/2022.csv")
teams = laliga_df["HomeTeam"].unique().tolist()
check_teams_in_wc(teams, league="ligue_1")

Bordeaux
St Etienne
Metz


In [24]:
from data_processing.featurise_2018 import featurise_europe_players

In [25]:
leagues_df = {
    "laliga": {
        2022: pd.read_csv("raw_data/la_liga/2022.csv"),
    },
    "bundesliga": {
        2022: pd.read_csv("raw_data/bundesliga/2022.csv"),
    },
    "serie_a": {
        2022: pd.read_csv("raw_data/serie_a/2022.csv"),
    },
    "ligue_1": {
        2022: pd.read_csv("raw_data/ligue_1/2022.csv"),
    }
}

In [46]:
with open("raw_data/domestic_top_scorers.json", "r") as f:
    domestic_top_scorers = json.load(f)

with open("raw_data/domestic_league_winners.json", "r") as f:
    league_winners = json.load(f)

prem_teams = ['Brentford',
 'Manchester Utd',
 'Leicester City',
 'Burnley',
 'Chelsea',
 'Watford',
 'Everton',
 'Norwich City',
 'Newcastle Utd',
 'Tottenham',
 'Liverpool',
 'Aston Villa',
 'Manchester City',
 'Leeds United',
 'Crystal Palace',
 'Brighton',
 'Wolves',
 'Southampton',
 'Arsenal',
 'West Ham']


countries = squad_df["Country"].unique().tolist()
results = {}
top_scorers = []
for _, scorers in domestic_top_scorers["2022"].items():
    top_scorers.extend(scorers)

winning_clubs = []
for _, teams in league_winners["2022"].items():
    winning_clubs.extend(teams)

for c in countries: 
    is_home_nation = 1 if c == "Qatar" else 0
    results[c] = {"is_home_nation": is_home_nation, "top_goalscorers_count": 0,"players_finishing_top_three_europe_domestic": 0, "premier_league_players": 0}
    c_df = squad_df[squad_df["Country"] == c]
    clubs = c_df["Club"].tolist()
    players = [p.replace("*(captain)*", "") for p in c_df["Player"].unique().tolist()]
    for club in clubs:
        if club in winning_clubs:
            results[c]["players_finishing_top_three_europe_domestic"] += 1
        featurise_europe_players(club, leagues_df, results[c], mapped_teams, 2022)
        for p_team in prem_teams:
            if mapped_teams["premier_league"].get(p_team, p_team) == club:
                results[c]["premier_league_players"] += 1
                continue
    for p in players:
        if p in top_scorers:
            results[c]["top_goalscorers_count"] += 1
    results[c]["n_unique_clubs"] = len(set(clubs))



In [47]:
results["England"]

{'is_home_nation': 0,
 'top_goalscorers_count': 1,
 'players_finishing_top_three_europe_domestic': 11,
 'premier_league_players': 25,
 'laliga_players': 0,
 'serie_a_players': 0,
 'bundesliga_players': 1,
 'ligue_1_players': 0,
 'n_unique_clubs': 11}

In [40]:
teams = squad_df[squad_df["Country"] == "England"]["Club"].tolist()

In [ ]:


winning_clubs = []
for _, teams in league_winners["2022"].items():
    winning_clubs.extend(teams)

all_clubs = squad_df["Club"].unique().tolist()

for club in winning_clubs:
    if club not in all_clubs:
        print(club)